In [2]:
# ========================
# PARTE 0: INSTALAÇÕES
# ========================
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, zipfile
from datetime import datetime

# ========================
# PARTE 2: CONFIG SELENIUM
# ========================
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ========================
# PARTE 3: TERMOS E API ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = ["pisos", "revestimentos", "porcelanatos"]

def buscar_links_por_categoria(termos, max_paginas=50, limite_por_termo=10):
    links = set()
    for termo in termos:
        print(f"🔍 Buscando: {termo}")
        contador = 0
        for pagina in range(max_paginas):
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = { "params": f"query={termo}&page={pagina}&hitsPerPage=100" }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200: break
            data = res.json()
            for item in data.get("hits", []):
                relative_url = item.get("url")
                if relative_url:
                    full_url = f"https://www.leroymerlin.com.br{relative_url}" if not relative_url.startswith("http") else relative_url
                    if full_url not in links:
                        links.add(full_url)
                        contador += 1
                        if contador >= limite_por_termo:
                            break
            if contador >= limite_por_termo: break
    return links

print("🔍 Coletando links...")
todos_links = buscar_links_por_categoria(termos_busca)
print(f"✅ Total de links: {len(todos_links)}\n")

# ========================
# PARTE 4: EXTRAÇÃO DE DADOS
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    preco_txt = None
    seletores = [
        '[data-cy="product-price"]',
        '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]',
        '.price__sales',
        '.value__integer',
        '.price',
        '.price__value'
    ]
    for sel in seletores:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text and text.lower() != "grátis":
                preco_txt = text
                break
    if preco_txt:
        try:
            preco_val = float(preco_txt.replace("R$", "").replace("/m²", "").replace(" ", "").replace(".", "").replace(",", "."))
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = "N/A"
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlin"
    return dados

# ========================
# PARTE 5: SALVAR PLANILHA DIRETO NO DRIVE
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = "/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY"
os.makedirs(base_drive, exist_ok=True)

arquivo_excel = os.path.join(base_drive, f"leroymerelin_{data_nome}.xlsx")
arquivo_csv   = os.path.join(base_drive, f"leroymerelin_{data_nome}.csv")

df = pd.DataFrame(todos)
df.drop(columns=[c for c in df.columns if c in ['Erro']], inplace=True, errors='ignore')
colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
df = df[colunas]
df.to_excel(arquivo_excel, index=False)
df.to_csv(arquivo_csv, index=False, sep=";")

print(f"✅ Excel salvo em: {arquivo_excel}")
print(f"✅ CSV salvo em: {arquivo_csv}")

# ========================
# PARTE 6: BAIXAR IMAGENS NO DRIVE
# ========================
pasta_imagens = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}")
os.makedirs(pasta_imagens, exist_ok=True)

for _, r in df.iterrows():
    url_img = r['Imagem']
    cod = r['Código']
    nome_arquivo = f"{cod}.png"
    caminho = os.path.join(pasta_imagens, nome_arquivo)

    if url_img and url_img != "N/A" and not os.path.exists(caminho):
        for _ in range(3):
            try:
                resp = requests.get(url_img, headers={"User-Agent": "Mozilla/5.0"}, timeout=10, stream=True)
                if resp.status_code == 200:
                    with open(caminho, "wb") as f:
                        for cb in resp.iter_content(1024):
                            f.write(cb)
                    print(f"✅ Baixada: {nome_arquivo}")
                    break
            except Exception as e:
                print(f"⚠️ Falha para {cod}: {e}")
                time.sleep(2)
    else:
        print(f"⏩ Já existe: {nome_arquivo}")

# Compactar imagens no mesmo local
zipname = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}.zip")
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(pasta_imagens):
        z.write(os.path.join(pasta_imagens, fn), fn)

print(f"📦 Imagens zipadas em: {zipname}")


# ========================
# PARTE 6: BAIXAR IMAGENS NO DRIVE
# ========================
base_drive = "/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY"
os.makedirs(base_drive, exist_ok=True)
pasta_imagens = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}")
os.makedirs(pasta_imagens, exist_ok=True)

for _, r in df.iterrows():
    url_img = r['Imagem']
    cod = r['Código']
    nome_arquivo = f"{cod}.png"
    caminho = os.path.join(pasta_imagens, nome_arquivo)

    if url_img and url_img != "N/A" and not os.path.exists(caminho):
        for _ in range(3):
            try:
                resp = requests.get(url_img, headers={"User-Agent": "Mozilla/5.0"}, timeout=10, stream=True)
                if resp.status_code == 200:
                    with open(caminho, "wb") as f:
                        for cb in resp.iter_content(1024):
                            f.write(cb)
                    print(f"✅ Baixada: {nome_arquivo}")
                    break
            except Exception as e:
                time.sleep(2)
    else:
        print(f"⏩ Já existe: {nome_arquivo}")

# Compactar imagens
zipname = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}.zip")
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(pasta_imagens):
        z.write(os.path.join(pasta_imagens, fn), fn)

print(f"📦 Imagens zipadas em: {zipname}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔍 Coletando links...
🔍 Buscando: pisos
🔍 Buscando: revestimentos
🔍 Buscando: porcelanatos
✅ Total de links: 30

✅ Excel salvo em: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/leroymerelin_2025-07-12.xlsx
✅ CSV salvo em: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/leroymerelin_2025-07-12.csv
⏩ Já existe: 89741211.png
⏩ Já existe: 92220324.png
⏩ Já existe: 92302714.png
⏩ Já existe: 91996856.png
⏩ Já existe: 92235682.png
⏩ Já existe: 92112181.png
⏩ Já existe: 92343524.png
⏩ Já existe: 91993664.png
⏩ Já existe: 91700182.png
⏩ Já existe: 92082144.png
⏩ Já existe: 92082123.png
⏩ Já existe: 92115975.png
⏩ Já existe: 92098860.png
⏩ Já existe: 92303141.png
⏩ Já existe: 92313781.png
⏩ Já 